In [ ]:
import scanpy as sc
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import os

# Load
tm = sc.read("tabula-muris-senis-facs-official-raw-obj.h5ad")

# Get top cell types by count, drop the 1st (most abundant), keep next 10
top_counts = tm.obs["cell_ontology_class"].value_counts()
print(top_counts.head(30))  # inspect before subsetting

selected_types = top_counts.index[0:30].tolist()  # take top 30
print(f"\nSelected cell types:")
for ct in selected_types:
    print(f"  {ct}: {top_counts[ct]} cells")


In [ ]:
mask = tm.obs["cell_ontology_class"].isin(selected_types)
tm_sub = tm[mask].copy()
print(f"Cells: {tm_sub.n_obs}, Genes: {tm_sub.n_vars}")


mmwrite("./counts.mtx", csr_matrix(tm_sub.X))

tm_sub.obs.index.to_series().to_csv(
    "./barcodes.csv", 
    index=False, 
    header=True
)

tm_sub.var.index.to_series().to_csv(
    "./genes.csv", 
    index=False, 
    header=True
)

tm_sub.obs.to_csv("./metadata.csv")

# Verify
print(pd.read_csv("./barcodes.csv").shape)
print(pd.read_csv("./genes.csv").shape)